# HW3 Part 2




**Course Code:**

**Group number:**

**Student Name:**

**Student ID:**

# Turn Continuity Classification


**Task**: Binary classification — predict whether a spoken turn is **Complete (1)** or **Incomplete (0)**.

**Metric**: Macro-F1 Score.

| Label | Meaning |
|---|---|
| 1 | **Complete** — semantic intent is finished; system can respond |
| 0 | **Incomplete** — intent is unfinished; system should keep listening |


## 0. Environment Setup & Data Loading

In [4]:
# Phase 2 only requires the standard scientific Python stack:
# pandas, numpy, scikit-learn. These are typically pre-installed.
# Uncomment the next line if you need to install them in a clean environment.
# !pip install -q pandas numpy scikit-learn


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import LinearSVC
from sklearn.utils import resample

print("All imports successful!")


In [6]:
# Resolve train.csv / test.csv whether the notebook runs from part2/ or repo root.
def find_csv(name: str) -> Path:
    candidates = [Path(name), Path('part2') / name, Path('..') / name]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(
        f"Could not locate {name}. Tried: {[str(p) for p in candidates]}"
    )

train_path = find_csv('train.csv')
test_path = find_csv('test.csv')
print(f"train: {train_path.resolve()}")
print(f"test : {test_path.resolve()}")

train_df = pd.read_csv(train_path)
public_test_df = pd.read_csv(test_path)

print(f"\nTrain size: {len(train_df)}  cols: {list(train_df.columns)}")
print(f"Test  size: {len(public_test_df)}  cols: {list(public_test_df.columns)}")
if 'label' in train_df.columns:
    print('\nLabel distribution:')
    print(train_df['label'].value_counts())
display(train_df.head())


train: C:\Coding\Intro_to_AI\project1\part2\train.csv
test : C:\Coding\Intro_to_AI\project1\part2\test.csv

Train size: 1302  cols: ['id', 'content', 'label']
Test  size: 500  cols: ['id', 'content']

Label distribution:
label
1    837
0    465
Name: count, dtype: int64


,id,content,label
0,1166,i think we should consider the actually the cl...,1
1,1127,Can you reset my password? My account password...,1
2,1240,im wondering if we need to no wait the test ca...,1
3,1853,"This code needs to be reviewed by tomorrow, do...",0
4,194,"This homework is taking forever, btw the new s...",0


In [7]:
# Standardize: rename text column to 'text', ensure 'id' exists.
TEXT_COL_CANDIDATES = ['text', 'content', 'utterance', 'sentence']

def standardize(df: pd.DataFrame) -> pd.DataFrame:
    text_col = next((c for c in TEXT_COL_CANDIDATES if c in df.columns), None)
    if text_col is None:
        raise KeyError(f"No text column found in {df.columns.tolist()}")
    if text_col != 'text':
        df = df.rename(columns={text_col: 'text'})
    if 'id' not in df.columns:
        df = df.reset_index(drop=True)
        df['id'] = df.index
    return df

train_df = standardize(train_df)
public_test_df = standardize(public_test_df)

print('Standardization complete.')
print(f"Train cols: {train_df.columns.tolist()}")
print(f"Test  cols: {public_test_df.columns.tolist()}")

if 'label' in train_df.columns:
    counts = train_df['label'].value_counts()
    print(f"\nLabel distribution (train):\n{counts}")
    if counts.min() > 0:
        print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.2f}")


Standardization complete.
Train cols: ['id', 'text', 'label']
Test  cols: ['id', 'text']

Label distribution (train):
label
1    837
0    465
Name: count, dtype: int64
Imbalance ratio (max/min): 1.80


## Part 1: Data Balancing

You must implement and compare two methods:
1. **Basic (Required)**: Random Over-sampling.
2. **Advanced (Choose 1+)**: EDA, Back-translation, SMOTE, or Cost-Sensitive Learning.

In [8]:
def perform_balancing(df: pd.DataFrame, method: str = 'random', random_state: int = 42) -> pd.DataFrame:
    """Phase 2 balancing.

    method='random' -> Random Over-sampling: duplicate minority class samples until 1:1.
    method='none'   -> return the dataframe unchanged.

    Cost-sensitive learning (class_weight='balanced') is handled at model fit time,
    not here, so it can be compared against random over-sampling fairly.
    """
    if method in ('none', None):
        return df.reset_index(drop=True)

    if method == 'random':
        counts = df['label'].value_counts()
        majority_label = counts.idxmax()
        minority_label = counts.idxmin()
        df_majority = df[df['label'] == majority_label]
        df_minority = df[df['label'] == minority_label]

        df_minority_upsampled = resample(
            df_minority,
            replace=True,
            n_samples=len(df_majority),
            random_state=random_state,
        )
        df_balanced = pd.concat([df_majority, df_minority_upsampled], axis=0)
        df_balanced = df_balanced.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
        return df_balanced

    raise ValueError(f"Unknown balancing method: {method}")


# Smoke test
sample_balanced = perform_balancing(train_df, method='random')
print('After Random Over-sampling:')
print(sample_balanced['label'].value_counts())


After Random Over-sampling:
label
0    837
1    837
Name: count, dtype: int64


## Part 2: Baseline Classifier (TF-IDF + SVM)

Establish a baseline. Use Macro-F1 as your primary metric.

In [9]:
# Optional text preprocessing hook. Phase 2 baseline uses raw text directly.
# Reserved as a Phase 3 extension point (lemmatization, stop-word removal, etc.).
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    return text.strip()


In [10]:
def get_model(model_type: str = 'svm', class_weight=None, C: float = 1.0, random_state: int = 42):
    """Phase 2 model factory.

    'svm' returns LinearSVC. Pass class_weight='balanced' for the cost-sensitive variant.
    LinearSVC is faster and more deterministic than kernel SVC on this dataset while
    remaining a Support Vector Machine.
    """
    if model_type == 'svm':
        return LinearSVC(
            C=C,
            class_weight=class_weight,
            random_state=random_state,
            max_iter=5000,
        )
    raise ValueError(f"Unknown model_type: {model_type}")


In [11]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label'],
)

print(f"Train subset : {len(train_data)} samples")
print(f"Val   subset : {len(val_data)} samples")
print('Train label distribution:')
print(train_data['label'].value_counts())
print('Val label distribution:')
print(val_data['label'].value_counts())


Train subset : 1041 samples
Val   subset : 261 samples
Train label distribution:
label
1    669
0    372
Name: count, dtype: int64
Val label distribution:
label
1    168
0     93
Name: count, dtype: int64


In [12]:
# Fair comparison: same TF-IDF + LinearSVC across three settings.
# Same train/val split, same feature config, only the balancing strategy varies.

TFIDF_PARAMS = dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True)


def run_experiment(name: str, train_data, val_data, *, balance: str, class_weight):
    train_used = perform_balancing(train_data, method=balance)

    vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
    X_train = vectorizer.fit_transform(train_used['text'].astype(str))
    y_train = train_used['label'].values

    clf = get_model('svm', class_weight=class_weight)
    clf.fit(X_train, y_train)

    X_val = vectorizer.transform(val_data['text'].astype(str))
    y_val = val_data['label'].values
    y_pred = clf.predict(X_val)

    macro_f1 = f1_score(y_val, y_pred, average='macro')
    print(f"\n=== {name} ===")
    print(f"Train rows used: {len(train_used)}  |  TF-IDF vocab: {len(vectorizer.vocabulary_)}")
    print(classification_report(y_val, y_pred, target_names=['Incomplete(0)', 'Complete(1)']))
    print(f"Macro-F1: {macro_f1:.4f}")
    return {'name': name, 'macro_f1': macro_f1}


results = []
results.append(run_experiment(
    'Baseline (no balancing)',
    train_data, val_data, balance='none', class_weight=None,
))
results.append(run_experiment(
    'Random Over-sampling',
    train_data, val_data, balance='random', class_weight=None,
))
results.append(run_experiment(
    'Cost-Sensitive (class_weight=balanced)',
    train_data, val_data, balance='none', class_weight='balanced',
))

summary = pd.DataFrame([{'method': r['name'], 'macro_f1': r['macro_f1']} for r in results])
print('\n=== Summary ===')
print(summary.to_string(index=False))

best = max(results, key=lambda r: r['macro_f1'])
print(f"\nBest balancing strategy on the validation split: {best['name']}  (macro_f1={best['macro_f1']:.4f})")



=== Baseline (no balancing) ===
Train rows used: 1041  |  TF-IDF vocab: 2168
               precision    recall  f1-score   support

Incomplete(0)       0.78      0.67      0.72        93
  Complete(1)       0.83      0.90      0.86       168

     accuracy                           0.82       261
    macro avg       0.81      0.78      0.79       261
 weighted avg       0.81      0.82      0.81       261

Macro-F1: 0.7919

=== Random Over-sampling ===
Train rows used: 1338  |  TF-IDF vocab: 3028
               precision    recall  f1-score   support

Incomplete(0)       0.80      0.61      0.70        93
  Complete(1)       0.81      0.92      0.86       168

     accuracy                           0.81       261
    macro avg       0.81      0.76      0.78       261
 weighted avg       0.81      0.81      0.80       261

Macro-F1: 0.7777

=== Cost-Sensitive (class_weight=balanced) ===
Train rows used: 1041  |  TF-IDF vocab: 2168
               precision    recall  f1-score   support

## Part 3: Enhancement (Stratified K-Fold + TF-IDF/Model sweep + Word+Char Ensemble)


In [ ]:
# Phase 3 enhancement: Stratified K-Fold + compact hyperparameter sweep.
# Same Macro-F1 metric, same fixed seed for fold splits across all configs => fair ranking.

PHASE3_N_SPLITS = 5
PHASE3_SEED = 42


def make_vec(kind, ngram_range, min_df=2, max_features=None, sublinear_tf=True):
    """Factory for word / char_wb TF-IDF vectorizers."""
    return TfidfVectorizer(
        analyzer='char_wb' if kind == 'char' else 'word',
        ngram_range=ngram_range,
        min_df=min_df,
        max_features=max_features,
        sublinear_tf=sublinear_tf,
    )


def make_svm(C, class_weight=None):
    return LinearSVC(C=C, class_weight=class_weight, random_state=PHASE3_SEED, max_iter=8000)


def make_lr(C, class_weight=None):
    return LogisticRegression(
        C=C,
        class_weight=class_weight,
        random_state=PHASE3_SEED,
        max_iter=2000,
        solver='liblinear',
    )


CONFIGS = [
    # word n-gram TF-IDF + LinearSVC
    {'name': 'word(1,2) + SVC C=1',         'kind': 'word', 'vec': lambda: make_vec('word', (1, 2)), 'model': lambda: make_svm(1.0),               'balance': 'none'},
    {'name': 'word(1,2) + SVC C=2',         'kind': 'word', 'vec': lambda: make_vec('word', (1, 2)), 'model': lambda: make_svm(2.0),               'balance': 'none'},
    {'name': 'word(1,3) + SVC C=1',         'kind': 'word', 'vec': lambda: make_vec('word', (1, 3)), 'model': lambda: make_svm(1.0),               'balance': 'none'},
    {'name': 'word(1,2) + SVC C=1 bal',     'kind': 'word', 'vec': lambda: make_vec('word', (1, 2)), 'model': lambda: make_svm(1.0, 'balanced'),   'balance': 'none'},
    {'name': 'word(1,2) + SVC C=1 oversmp', 'kind': 'word', 'vec': lambda: make_vec('word', (1, 2)), 'model': lambda: make_svm(1.0),               'balance': 'random'},
    # char n-gram TF-IDF + LinearSVC
    {'name': 'char(3,5) + SVC C=1',         'kind': 'char', 'vec': lambda: make_vec('char', (3, 5)), 'model': lambda: make_svm(1.0),               'balance': 'none'},
    {'name': 'char(3,5) + SVC C=2',         'kind': 'char', 'vec': lambda: make_vec('char', (3, 5)), 'model': lambda: make_svm(2.0),               'balance': 'none'},
    {'name': 'char(4,6) + SVC C=1',         'kind': 'char', 'vec': lambda: make_vec('char', (4, 6)), 'model': lambda: make_svm(1.0),               'balance': 'none'},
    {'name': 'char(3,5) + SVC C=1 bal',     'kind': 'char', 'vec': lambda: make_vec('char', (3, 5)), 'model': lambda: make_svm(1.0, 'balanced'),   'balance': 'none'},
    # LogisticRegression heads
    {'name': 'word(1,2) + LR C=1',          'kind': 'word', 'vec': lambda: make_vec('word', (1, 2)), 'model': lambda: make_lr(1.0),                'balance': 'none'},
    {'name': 'word(1,2) + LR C=2 bal',      'kind': 'word', 'vec': lambda: make_vec('word', (1, 2)), 'model': lambda: make_lr(2.0, 'balanced'),    'balance': 'none'},
    {'name': 'char(3,5) + LR C=2',          'kind': 'char', 'vec': lambda: make_vec('char', (3, 5)), 'model': lambda: make_lr(2.0),                'balance': 'none'},
]


def evaluate_config(cfg, df, n_splits=PHASE3_N_SPLITS):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=PHASE3_SEED)
    fold_scores = []
    X_text = df['text'].astype(str).values
    y = df['label'].values
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_text, y)):
        tr = df.iloc[tr_idx]
        va = df.iloc[va_idx]
        if cfg['balance'] == 'random':
            tr = perform_balancing(tr, method='random', random_state=PHASE3_SEED + fold)
        vec = cfg['vec']()
        X_tr = vec.fit_transform(tr['text'].astype(str).values)
        y_tr = tr['label'].values
        X_va = vec.transform(va['text'].astype(str).values)
        y_va = va['label'].values
        clf = cfg['model']()
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_va)
        fold_scores.append(f1_score(y_va, y_pred, average='macro'))
    return fold_scores


phase3_records = []
for cfg in CONFIGS:
    fold_scores = evaluate_config(cfg, train_df)
    phase3_records.append({
        'config_name': cfg['name'],
        'vectorizer_type': 'word' if cfg['kind'] == 'word' else 'char_wb',
        'model_type': 'LinearSVC' if 'SVC' in cfg['name'] else 'LogisticRegression',
        'balance': cfg['balance'],
        'mean_macro_f1': float(np.mean(fold_scores)),
        'std_macro_f1': float(np.std(fold_scores)),
        'fold_scores': [round(s, 4) for s in fold_scores],
    })
    print(f"{cfg['name']:32s}  mean={np.mean(fold_scores):.4f}  std={np.std(fold_scores):.4f}")

phase3_df = pd.DataFrame(phase3_records).sort_values('mean_macro_f1', ascending=False).reset_index(drop=True)
print('\n=== Phase 3 single-model K-Fold results (sorted by mean Macro-F1) ===')
print(phase3_df.to_string(index=False))


In [ ]:
# Word + Char ensemble: combine z-scored decision_function from the best word and char configs.
# This averages two complementary views (lexical word features and sub-word morphology).

config_by_name = {cfg['name']: cfg for cfg in CONFIGS}


def best_of_kind(kind):
    for _, row in phase3_df.iterrows():
        cfg = config_by_name[row['config_name']]
        if cfg['kind'] == kind:
            return cfg
    return None


best_word_cfg = best_of_kind('word')
best_char_cfg = best_of_kind('char')
print(f"Best word config: {best_word_cfg['name']}")
print(f"Best char config: {best_char_cfg['name']}")


def decision_scores(model, X):
    """Signed decision score; works for both LinearSVC and LogisticRegression in binary."""
    return model.decision_function(X)


def zscore(arr):
    arr = np.asarray(arr, dtype=np.float64)
    mu = arr.mean()
    sigma = arr.std() + 1e-12
    return (arr - mu) / sigma, mu, sigma


def cv_ensemble(word_cfg, char_cfg, weights, df, n_splits=PHASE3_N_SPLITS):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=PHASE3_SEED)
    scores = []
    X_text = df['text'].astype(str).values
    y = df['label'].values
    for tr_idx, va_idx in skf.split(X_text, y):
        tr = df.iloc[tr_idx]
        va = df.iloc[va_idx]

        word_vec = word_cfg['vec'](); word_clf = word_cfg['model']()
        word_clf.fit(word_vec.fit_transform(tr['text'].astype(str).values), tr['label'].values)
        wv_tr = decision_scores(word_clf, word_vec.transform(tr['text'].astype(str).values))
        wv_va = decision_scores(word_clf, word_vec.transform(va['text'].astype(str).values))

        char_vec = char_cfg['vec'](); char_clf = char_cfg['model']()
        char_clf.fit(char_vec.fit_transform(tr['text'].astype(str).values), tr['label'].values)
        cv_tr = decision_scores(char_clf, char_vec.transform(tr['text'].astype(str).values))
        cv_va = decision_scores(char_clf, char_vec.transform(va['text'].astype(str).values))

        # Standardize using TRAIN-fold stats so the inference recipe matches.
        _, wmu, wsig = zscore(wv_tr)
        _, cmu, csig = zscore(cv_tr)
        wsc = (wv_va - wmu) / wsig
        csc = (cv_va - cmu) / csig

        ww, wc = weights
        combined = ww * wsc + wc * csc
        y_pred = (combined > 0).astype(int)
        scores.append(f1_score(va['label'].values, y_pred, average='macro'))
    return scores


ENSEMBLE_WEIGHTS = [(0.3, 0.7), (0.5, 0.5), (0.7, 0.3)]
ensemble_records = []
for w in ENSEMBLE_WEIGHTS:
    scores = cv_ensemble(best_word_cfg, best_char_cfg, w, train_df)
    ensemble_records.append({
        'config_name': f'ensemble word*{w[0]} + char*{w[1]}',
        'vectorizer_type': 'word+char_wb',
        'model_type': 'ensemble',
        'balance': 'none',
        'mean_macro_f1': float(np.mean(scores)),
        'std_macro_f1': float(np.std(scores)),
        'fold_scores': [round(s, 4) for s in scores],
    })
    print(f"ensemble w={w}: mean={np.mean(scores):.4f}  std={np.std(scores):.4f}")

ensemble_df = pd.DataFrame(ensemble_records).sort_values('mean_macro_f1', ascending=False).reset_index(drop=True)
print('\n=== Ensemble K-Fold results ===')
print(ensemble_df.to_string(index=False))


In [ ]:
# Combined Phase 3 leaderboard (single configs + ensembles), sorted by mean Macro-F1.
all_phase3_df = pd.concat([phase3_df, ensemble_df], ignore_index=True)
all_phase3_df = all_phase3_df.sort_values('mean_macro_f1', ascending=False).reset_index(drop=True)

print('=== Phase 3 combined results ===')
print(all_phase3_df.to_string(index=False))

best_overall = all_phase3_df.iloc[0]
best_single = phase3_df.iloc[0]
best_ensemble = ensemble_df.iloc[0]
print(f"\nBest overall:  {best_overall['config_name']}  mean Macro-F1={best_overall['mean_macro_f1']:.4f}")
print(f"Best single:   {best_single['config_name']}  mean Macro-F1={best_single['mean_macro_f1']:.4f}")
print(f"Best ensemble: {best_ensemble['config_name']}  mean Macro-F1={best_ensemble['mean_macro_f1']:.4f}")


## Part 4: Final Submission

Train on the full dataset using your best found configuration and generate `submission.csv`.

In [ ]:
# Fit the chosen single model AND the word+char ensemble on the full training set
# so we can produce both submissions and let Kaggle pick the winner.

# 1) Best single configuration (highest mean Macro-F1 in single-model sweep).
best_single_cfg = config_by_name[best_single['config_name']]
print(f"Fitting best single config on full train: {best_single_cfg['name']}")

if best_single_cfg['balance'] == 'random':
    full_single_df = perform_balancing(train_df, method='random', random_state=PHASE3_SEED)
else:
    full_single_df = train_df

best_single_vec = best_single_cfg['vec']()
X_full_single = best_single_vec.fit_transform(full_single_df['text'].astype(str).values)
y_full_single = full_single_df['label'].values
best_single_model = best_single_cfg['model']()
best_single_model.fit(X_full_single, y_full_single)
print(f"  Trained on {X_full_single.shape[0]} rows; vocab={len(best_single_vec.vocabulary_)}")

# 2) Best ensemble: refit best word + best char on full train, recompute z-score stats on train.
print(f"\nFitting word+char ensemble on full train using:")
print(f"  word: {best_word_cfg['name']}")
print(f"  char: {best_char_cfg['name']}")

word_vec_full = best_word_cfg['vec']()
word_model_full = best_word_cfg['model']()
word_model_full.fit(word_vec_full.fit_transform(train_df['text'].astype(str).values), train_df['label'].values)
word_train_scores = decision_scores(word_model_full, word_vec_full.transform(train_df['text'].astype(str).values))
_, word_mu, word_sigma = zscore(word_train_scores)

char_vec_full = best_char_cfg['vec']()
char_model_full = best_char_cfg['model']()
char_model_full.fit(char_vec_full.fit_transform(train_df['text'].astype(str).values), train_df['label'].values)
char_train_scores = decision_scores(char_model_full, char_vec_full.transform(train_df['text'].astype(str).values))
_, char_mu, char_sigma = zscore(char_train_scores)

# Use the best ensemble weight from CV.
best_ensemble_name = best_ensemble['config_name']
ensemble_weight_word = float(best_ensemble_name.split('word*')[1].split(' ')[0])
ensemble_weight_char = float(best_ensemble_name.split('char*')[1])
print(f"  Ensemble weights: word={ensemble_weight_word}  char={ensemble_weight_char}")


In [ ]:
def write_submission(test_df, predictions, output_name):
    submission = pd.DataFrame({
        'id': test_df['id'].values,
        'label': np.asarray(predictions, dtype=int),
    })
    submission.to_csv(output_name, index=False)
    print(f"Saved -> {output_name}")
    print(f"  prediction distribution: {submission['label'].value_counts().to_dict()}")
    return submission


test_text = public_test_df['text'].astype(str).values

# 1) Best single submission.
single_pred = best_single_model.predict(best_single_vec.transform(test_text))
single_submission = write_submission(public_test_df, single_pred, 'submission_phase3_best_single.csv')

# 2) Word+char ensemble submission.
word_test_scores = decision_scores(word_model_full, word_vec_full.transform(test_text))
char_test_scores = decision_scores(char_model_full, char_vec_full.transform(test_text))
word_test_z = (word_test_scores - word_mu) / word_sigma
char_test_z = (char_test_scores - char_mu) / char_sigma
combined_test = ensemble_weight_word * word_test_z + ensemble_weight_char * char_test_z
ensemble_pred = (combined_test > 0).astype(int)
ensemble_submission = write_submission(public_test_df, ensemble_pred, 'submission_phase3_word_char_ensemble.csv')

# 3) Primary submission.csv = whichever has higher CV Macro-F1.
if best_overall['model_type'] == 'ensemble':
    primary_pred = ensemble_pred
    primary_label = 'ensemble'
else:
    primary_pred = single_pred
    primary_label = 'single'
primary_submission = write_submission(public_test_df, primary_pred, 'submission.csv')
print(f"\nPrimary submission.csv chosen from: {primary_label}  ({best_overall['config_name']})")
display(primary_submission.head(10))


In [16]:
# Download in Colab
try:
    from google.colab import files
    files.download('submission.csv')
    print("Download started!")
except ImportError:
    print("Not running in Colab — file saved locally as submission.csv")

Not running in Colab — file saved locally as submission.csv
